In [0]:
gold_table = "workspace.ecommerce.user_predictions_gold"

In [0]:
print("Current table history:")
spark.sql(f"DESCRIBE HISTORY {gold_table}").display()

In [0]:
# Load existing data
df = spark.table(gold_table)

# Simulate new batch (same schema)
new_data = df.limit(100)

# Append safely
new_data.write.mode("append").saveAsTable(gold_table)

print("New records appended successfully.")

In [0]:
print("Table history AFTER append")
spark.sql(f"DESCRIBE HISTORY {gold_table}").display()

In [0]:
# Get previous version
history_df = spark.sql(f"DESCRIBE HISTORY {gold_table}")

old_version = history_df.selectExpr("max(version)-1 as v").collect()[0]["v"]

print("Reading version:", old_version)

old_df = spark.read.option("versionAsOf", old_version).table(gold_table)
current_df = spark.table(gold_table)

print("Old count:", old_df.count())
print("Current count:", current_df.count())

In [0]:
diff_df = current_df.subtract(old_df)

print("New rows added:")
diff_df.display()

In [0]:
spark.sql(f"""
RESTORE TABLE {gold_table}
TO VERSION AS OF {old_version}
""")

print("Table restored to previous version.")

In [0]:
spark.sql("DESCRIBE HISTORY workspace.ecommerce.user_predictions_gold").display()